# 1. Train - simulated CRM data (`data/crm_simulated/`)

This notebook do same thing like `python -m src.train --data-dir data/crm_simulated` but step by step, it use the function from `src/train.py`.
Model get save into `reports/crm/simulated/final/models/` so the `2_evaluate.ipynb` and `3_predict.ipynb` can loading them. We dont touch any test row in here.

Kernel: using the `.venv-crm` enviroment (python 3.12).

Simulated data have to be in `data/crm_simulated/` first (run the script in `scripts/simulation/` for make it).
Result never go to `reports/crm/final/`. Full run take few minute, TabNet is the most slowest.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from src import train

## Settings

In [ ]:
DATA_DIR = ROOT / "data/crm_simulated"
QUICK = False
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

## 1. Load, validate and split data

Join the four CRM table together, build the pre-close features (18 normal one + 11 simulated deal-level one) and then split closed deal by engage date.
Training and validation only keeping deals which there outcome was already know before next period.

In [ ]:
out, manifest, dataset, split = train.start(OUTPUT, quick=QUICK, data_dir=DATA_DIR)
report = dataset.report
pd.Series({
    "closed deals (labeled)": report["rows"],
    "won": report["positives"],
    "lost": report["negatives"],
    "open deals": report["open_rows"],
    "scorable Engaging deals": report["scorable_open_rows"],
    "scorable deals missing an account": report["scorable_missing_account"],
    "features": len(dataset.feature_columns),
})

In [ ]:
pd.DataFrame(manifest["split"], index=["value"]).T

## 2. Fit the five model

Every cell fit one model on training row, choose it's threshold on validation (macro-F1)
and showing the validation metric.

In [ ]:
fitted = {}
COLUMNS = ["model", "roc_auc", "accuracy", "precision", "recall", "f1", "brier", "threshold"]

def show(name):
    return pd.DataFrame([fitted[name][2]])[COLUMNS].round(4)

fitted["dummy_prior"] = train.fit_one("dummy_prior", dataset, split, quick=QUICK)
show("dummy_prior")

In [ ]:
fitted["logistic_regression"] = train.fit_one("logistic_regression", dataset, split, quick=QUICK)
show("logistic_regression")

In [ ]:
fitted["random_forest"] = train.fit_one("random_forest", dataset, split, quick=QUICK)
show("random_forest")

In [ ]:
fitted["mlp"] = train.fit_one("mlp", dataset, split, quick=QUICK)
show("mlp")

In [ ]:
fitted["tabnet"] = train.fit_one("tabnet", dataset, split, quick=QUICK)
show("tabnet")

### Learning curve for neural model

Early stopping keep the epoch who have best validation ROC-AUC (the dash line).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, name in zip(axes, ["mlp", "tabnet"]):
    model = fitted[name][0]
    ax.plot(range(1, len(model.history["validation_auc"]) + 1), model.history["validation_auc"])
    ax.axvline(model.settings["best_epoch"], ls="--", color="gray")
    ax.set(title=name, xlabel="Epoch", ylabel="Validation ROC-AUC")
plt.tight_layout()

## 3. Compare on validation, select, calibrate and save

We picking the model with highest validation ROC-AUC (if it tie, lower Brier win and then by name).
After that a sigmoid calibration is fit on validation set around selected model, the model it self is freeze.

In [ ]:
pd.DataFrame([f[2] for f in fitted.values()])[COLUMNS].sort_values("roc_auc", ascending=False).round(4)

In [ ]:
trained = train.finish(out, manifest, dataset, split, fitted)
print("Selected:", trained.selected)
print("Raw threshold:", trained.thresholds[trained.selected], "-> calibrated:", round(trained.decision_threshold, 4))
print("Calibration slope / intercept:", round(trained.calibrated.slope, 4), round(trained.calibrated.intercept, 4))
sorted(p.name for p in out.models.iterdir())

Next go to `2_evaluate.ipynb` for test metric and after that `3_predict.ipynb` for get the scores.